In [ ]:
#这个是必须要有的
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

%env http_proxy=http://127.0.0.1:7890
%env https_proxy=http://127.0.0.1:7890
%env all_proxy=socks5://127.0.0.1:7890

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import json
from tqdm import tqdm
import re
os.environ["CUDA_VISIBLE_DEVICES"] = "2" 

# ===== 基础配置 =====
MODEL_PATH = ""
INPUT_JSONL = ""
OUTPUT_JSONL = ""
EVIDENCE_SEP = "\n\n---\n\n"  # 证据分隔符


tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto"
)

def build_stem(item: dict) -> str:
    rq = item.get("raw_question")
    if rq and isinstance(rq, str) and rq.strip():
        stem = rq.strip()
    else:
        stem = (item.get("question") or "Answer the multiple-choice question.").strip()
    if not stem.lower().strip().endswith("answer:"):
        stem += "\nAnswer:"
    return stem


def build_assistant_evidence(docs: list) -> str:
    if not docs:
        return "No retrieved documents were provided."
    chunks = [
        f"[{i+1}] Title: {d.get('title') or f'doc_{i}'}\n{d.get('text') or ''}"
        for i, d in enumerate(docs)
    ]
    prefix = (
        "The following retrieved documents are provided verbatim before the question. "
        "Answer the upcoming multiple-choice question with only the final option letter "
        "(e.g., A, B, C, or D) without explanations."
    )
    return prefix + "\n\n" + EVIDENCE_SEP.join(chunks)


dataset = []
with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            dataset.append(json.loads(line))
print(f"成功加载数据集: {INPUT_JSONL}，共 {len(dataset)} 条数据。")

results = []


for item in tqdm(dataset, desc="Processing Dataset"):
    stem = build_stem(item)                 # 题干（以 Answer: 结尾）
    docs = item.get("docs", []) or []       # 全量上下文
    assistant_ctx = build_assistant_evidence(docs)


    messages = [
        {"role": "system", "content": "You are a helpful assistant. Your answer must be only the letter of the correct option (e.g., A, B, C, or D)."},
        {"role": "assistant", "content": assistant_ctx},
        {"role": "user", "content": stem},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


    inputs = tokenizer(prompt, return_tensors="pt", truncation=False).to(model.device)
    input_len = inputs["input_ids"].shape[1]

    outputs_ids = model.generate(
        **inputs,
        max_new_tokens=20,  # 仅需一个字母，给小一些
        do_sample=False,    # 贪婪
        custom_generate="",
        trust_remote_code=True,
        dola_layers="high",
        repetition_penalty=1.2,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id,
    )

    gen_ids = outputs_ids[0][input_len:]
    gen_text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()


    m = re.search(r"\b([ABCD])\b", gen_text, flags=re.IGNORECASE)
    final_choice = m.group(1).upper() if m else gen_text

    results.append({
        "question_prompted": stem,                  
        "model_answer": final_choice,
        "raw_model_output": gen_text,
        "ground_truths": item.get("ground_truths", [])
    })

with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("\n处理完成！")
print(f"所有结果已成功保存至: {OUTPUT_JSONL}")
